In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. 时间驱动变换器 (Time-Varying Transformer)
# ==========================================
class TimeVaryingTransformer(nn.Module):
    """
    【核心逻辑】：将时间步 t 作为 Value。
    【物理意义】：利用当前信号(Q)与粗调基准(K)的匹配度，去决定在当前时间(V=t)下，
                每个像素点应该分配多少“进化的动量”。
    """
    def __init__(self, num_leads=12, num_heads=4):
        super().__init__()
        self.num_leads = num_leads
        self.num_heads = num_heads
        self.head_dim = num_leads // num_heads

        # Q 和 K 依然处理 12 导联信号
        self.to_q = nn.Conv1d(num_leads, num_leads, kernel_size=1, groups=num_leads)
        self.to_k = nn.Conv1d(num_leads, num_leads, kernel_size=1, groups=num_leads)
        
        # 【关键点】：V 是时间 t。由于 t 是标量，我们要把它映射到导联空间
        self.to_v = nn.Linear(1, num_leads) 
        
        self.proj = nn.Conv1d(num_leads, num_leads, kernel_size=1)
        self.norm = nn.LayerNorm(num_leads)

    def forward(self, x_t, x_con, t):
        """
        x_t: [B, 12, L] - 当前流动状态
        x_con: [B, 12, L] - UNet 基准
        t: [B, 1] - 当前流时间 (0~1)
        """
        B, C, L = x_t.shape # C=12

        # --- Step 1: 提取 Q, K (来自信号) ---
        # [B, 12, L] -> [B, L, 12]
        q = self.to_q(x_t).permute(0, 2, 1) 
        k = self.to_k(x_con).permute(0, 2, 1)

        # --- Step 2: 构造 V (来自时间 t) ---
        # 我们要把 [B, 1] 的时间，扩展成每个采样点都能感知的形式
        # [B, 1] -> [B, 1, 12] -> [B, L, 12]
        # 物理意义：这一时刻全局的“进化速度”指令
        v = self.to_v(t).unsqueeze(1).repeat(1, L, 1) 

        # --- Step 3: 分头处理 (Multi-head) ---
        # [B, L, 12] -> [B, heads, L, head_dim]
        q = q.view(B, L, self.num_heads, -1).transpose(1, 2)
        k = k.view(B, L, self.num_heads, -1).transpose(1, 2)
        v = v.view(B, L, self.num_heads, -1).transpose(1, 2)

        # --- Step 4: 计算注意力矩阵 ---
        # attn 形状: [B, heads, L, L] 
        # 衡量当前信号每一个点与基准信号每一个点在结构上的“契合度”
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)

        # --- Step 5: 用结构契合度去加权“时间动量” ---
        # 结果形状: [B, L, 12] -> [B, 12, L]
        # 逻辑：契合度越高的地方，分配到的 $t$ 动量越大，跑得越快
        out = (attn @ v).transpose(1, 2).reshape(B, L, C).permute(0, 2, 1)
        
        out = self.proj(out)
        
        # 残差连接：x_t + 经过时间加权的修正量
        res = self.norm((out + x_t).permute(0, 2, 1)).permute(0, 2, 1)
        return res # [B, 12, L]

# ==========================================
# 2. 12导联精调速度场网络 (DGG-TimeFlow-Net)
# ==========================================
class TimeFlowVelocityNet(nn.Module):
    def __init__(self, num_leads=12):
        super().__init__()
        # FiLM 依然保留，作为全局的增益控制 [B, 12]
        self.t_mlp = nn.Sequential(
            nn.Linear(1, num_leads * 2), nn.SiLU(), nn.Linear(num_leads * 2, num_leads * 2)
        )
        
        # 注入时间 V 的 Transformer
        self.transformer = TimeVaryingTransformer(num_leads=num_leads)
        
        # 最终速度输出层
        self.final_conv = nn.Conv1d(num_leads, num_leads, kernel_size=3, padding=1, groups=num_leads)

    def forward(self, x_t, t, x_con):
        """
        x_t: [B, 12, L]
        t: [B, 1]
        x_con: [B, 12, L]
        """
        B, C, L = x_t.shape

        # 1. FiLM 调制 (全局节奏控制)
        t_params = self.t_mlp(t) # [B, 24]
        gamma, beta = torch.chunk(t_params, 2, dim=-1)
        
        # 广播到时间轴 L: [B, 12, 1]
        gamma, beta = gamma.unsqueeze(-1), beta.unsqueeze(-1)
        f_t = x_t * (1 + gamma) + beta 

        # 2. Transformer 局部导航 (将 t 作为 Value 注入)
        # v_feat 形状: [B, 12, L]
        v_feat = self.transformer(f_t, x_con, t)

        # 3. 输出速度场 v_t
        v_t = self.final_conv(v_feat)
        return v_t # [B, 12, L]

# ==========================================
# 3. 完整业务封装
# ==========================================
class ECGFlowSystem(nn.Module):
    def __init__(self, unet, velocity_net):
        super().__init__()
        self.unet = unet
        self.v_net = velocity_net

    def training_loss(self, x_masked, x_real):
        """
        训练：信号在 12 维空间内沿时间步演化
        """
        B, C, L = x_masked.shape
        
        # Step 1: 粗调基准
        with torch.no_grad():
            x_con = self.unet(x_masked) # [B, 12, L]
        
        # Step 2: 流路径
        t = torch.rand(B, 1, device=x_masked.device)
        t_img = t.view(B, 1, 1)
        # 从粗调到真实的线性路径
        x_t = (1 - t_img) * x_con + t_img * x_real
        
        # 目标速度: x_real - x_con
        target_v = x_real - x_con 
        
        # Step 3: 预测
        pred_v = self.v_net(x_t, t, x_con)
        return F.mse_loss(pred_v, target_v)

    @torch.no_grad()
    def reconstruct(self, x_masked, steps=20):
        """
        推理：两阶段精调
        """
        x_con = self.unet(x_masked) # [B, 12, L]
        
        x_t = x_con
        dt = 1.0 / steps
        for i in range(steps):
            t_val = torch.ones(x_con.shape[0], 1, device=x_con.device) * (i / steps)
            v_t = self.v_net(x_t, t_val, x_con)
            x_t = x_t + v_t * dt # 电压步进
            
        return x_t # [B, 12, L]